# Demand forecasting for weekly branch planning

**Question:** how many transactions should each branch prepare to serve
on each of the next seven days? The intended use is a staffing/preparation
pilot. Financial savings are not established by prediction accuracy.

This contribution extends `01_eda.ipynb`: Tuesday demand and growth justify
weekday/trend features; incomplete customer IDs discourage customer-level
targeting. Garden's standard-cost anomaly remains a separate audit action.
This extends notebook 03 from next-day prediction to seven-day weekly planning.

Install `requirements-week-ahead.txt` if dependencies are missing, select the Python kernel, and Run All.
No variables from another notebook are needed. No raw rows or customer IDs
are displayed or exported.

## How this extends the team submission

Notebooks 02–03 predict tomorrow; this contribution forecasts each day of the
next week together and refits weekly. It does not replace root-cause, offer,
product-stock or Garden profit-outlook analyses. Notebook 13 reuses notebook
04's hypothetical shortage/excess penalties on matching week-ahead forecasts.
The existing next-day XGBoost scores use a different protocol and are not a
controlled head-to-head comparison. June remains exploratory.


In [1]:
from pathlib import Path
import sys
import os
import json
import hashlib
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# Set this to a path if the team stores its CSV somewhere else.
DATA_OVERRIDE = os.environ.get("STORIES_DATA")
cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents)
             if (p / "src/stories_coffee/pipeline.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Copy src/stories_coffee/ alongside notebooks/ first.")
sys.path.insert(0, str(ROOT / "src"))
import stories_coffee
if Path(stories_coffee.__file__).resolve().parent != ROOT / "src/stories_coffee":
    raise RuntimeError("Another stories_coffee package is loaded. Restart the kernel.")
if DATA_OVERRIDE is not None:
    DATA = Path(DATA_OVERRIDE).expanduser()
    if not DATA.is_absolute():
        DATA = ROOT / DATA
else:
    candidates = [ROOT / p for p in (
        "data/raw/stories_transactions.csv", "data/stories_transactions.csv",
        "stories_transactions.csv") if (ROOT / p).is_file()]
    if len(candidates) != 1:
        raise FileNotFoundError("Set DATA_OVERRIDE: expected exactly one source CSV.")
    DATA = candidates[0]
if not DATA.is_file():
    raise FileNotFoundError(DATA)
OUT = ROOT / "artifacts/week_ahead_forecasting"
PUBLIC = ROOT / "reports/week_ahead_forecasting"
OUT.mkdir(parents=True, exist_ok=True)
print("Source:", DATA.relative_to(ROOT) if DATA.is_relative_to(ROOT) else DATA.name)
print("SHA256:", hashlib.sha256(DATA.read_bytes()).hexdigest())
print("Outputs: artifacts/week_ahead_forecasting/")

Source: stories_transactions.csv
SHA256: f1672e728c28295b2b802d878fd38a158ba039827b429355e8438ee0463237a1
Outputs: artifacts/week_ahead_forecasting/


## 1. Define the target before fitting

Count **distinct transaction IDs**, not product lines or units. The loader
checks missing fields and inconsistent branch/date metadata within a basket.
Exact duplicate product lines do not add visits. This differs from financial
and product-level EDA, where duplicate handling can affect totals.

**Assumptions:** missing branch-days stop fitting for investigation; branches remain open and
comparable; six months are insufficient to identify annual seasonality.
Sales can understate demand when a branch is closed or stocked out.

In [2]:
from stories_coffee.data import load_data
data = load_data(DATA)
display(data.daily.groupby("branch").agg(
    observed_days=("date", "size"), transactions=("transactions", "sum")))
print("History:", data.daily.date.min().date(), "to", data.end.date())

,observed_days,transactions
branch,,
Garden,181,11163
Harbour,181,16334
Uptown,181,6390


History: 2026-01-01 to 2026-06-30


## 2. Fit, select and test in date order

The original Poisson regression learns branch/weekday patterns and branch
growth. New candidates share weekday patterns across branches, regularize
differences, and test alternative growth/recency assumptions. Blending can
stabilize predictions. This is statistical machine learning; complexity
alone does not demonstrate value.

Eight March/April origins shortlist three of 24 candidates; four May origins
select a candidate or fixed blend against the regression baseline. Five June
origins evaluate it on 90 branch-days. Every fit excludes its forecast dates;
completed prior June weeks can enter subsequent training. June has been
inspected previously, so this is **exploratory**, not an untouched test.

Run the existing tested pipeline rather than duplicating its optimizer.
Notebook 13 adds a simpler seasonal-naive comparator on identical test rows.

In [3]:
from week_ahead_integration import verify_team_economics_source
verify_team_economics_source(ROOT)
from stories_coffee.pipeline import train
result = train(DATA, OUT)
display(pd.DataFrame(result["selected_recipe"].items(), columns=["component", "weight"]))
selection = json.loads((OUT / "selection.json").read_text())
display(pd.DataFrame(selection["validation_ranking"])[["method", "wape", "mae", "rmse"]])

,component,weight
0,baseline,0.5
1,pooled_0_log90_halfall,0.5


,method,wape,mae,rmse
0,half_baseline_devwinner,0.082917,5.800263,7.252683
1,pooled_0_log90_halfall,0.083014,5.807011,7.255985
2,pooled_0_log90_half56,0.083565,5.845594,7.324702
3,average_devtop3,0.083738,5.857654,7.328558
4,pooled_0.05_log90_halfall,0.084934,5.941346,7.437478
5,baseline,0.085139,5.955659,7.412722


## 3. Report errors honestly

WAPE = total absolute forecast error / total actual transactions. The
shorthand `100% − WAPE` is **not** the percentage of predictions exactly
correct. Compare MAE, RMSE and bias as well. The historical expected result
for the supplied source is about 91.17% versus 90.96% for the regression
baseline; the code below calculates rather than hard-codes the scores.

In [4]:
comparison = pd.DataFrame({"Regression baseline": result["baseline"],
                           "Selected forecast": result["selected"]}).T
comparison["100-WAPE (%)"] = 100 * (1 - comparison.wape)
display(comparison[["100-WAPE (%)", "wape", "mae", "rmse", "bias", "n"]])
lo, hi = 100 * np.array(result["bootstrap95_wape_reduction"])
print(f"WAPE gain: {result['wape_reduction_percentage_points']:.3f} percentage points")
print(f"Descriptive 95% gain interval: [{lo:.3f}, {hi:.3f}] percentage points")
print(f"Approximate DM/HAC p-value: {result['dm']['p_value_two_sided']:.4f}")
print("A small gain with an interval crossing zero does not establish superiority.")

,100-WAPE (%),wape,mae,rmse,bias,n
Regression baseline,90.955565,0.090444,6.940096,8.858719,0.004152,90.0
Selected forecast,91.172300,0.088277,6.773789,8.922356,-0.012520,90.0


WAPE gain: 0.217 percentage points
Descriptive 95% gain interval: [-0.133, 0.593] percentage points
Approximate DM/HAC p-value: 0.2076
A small gain with an interval crossing zero does not establish superiority.


## 4. Export a practical planning input

These are daily **transaction** forecasts issued weekly, not item orders or
hourly staffing requirements. The forecast starts after the source ends.
With the supplied file it covers July 1–7, 2026, regardless of today's date.

In [5]:
future = pd.read_csv(OUT / "tables/forecast_next_7_days.csv")
display(future.pivot(index="date", columns="branch", values="predicted").round(1))
print("Start:", result["next_forecast_origin"], "| source ends:", result["source_end"])

branch,Garden,Harbour,Uptown
date,,,
2026-07-01,75.2,113.7,42.5
2026-07-02,73.5,109.8,42.5
2026-07-03,80.6,121.1,45.8
2026-07-04,52.9,78.3,30.0
2026-07-05,63.7,95.1,36.2
2026-07-06,73.6,107.8,41.3
2026-07-07,117.1,173.2,65.7


Start: 2026-07-01 | source ends: 2026-06-30


## 5. Business recommendation and limits

Freeze this recipe and run a prospective staffing pilot against normal
planning, recording paid hours and service delays. Define transaction
capacity per staff-hour and service standards before assigning shifts.
Report payroll/service outcomes separately from forecast error. Hourly
sales, opening hours, staffing and promotions are missing inputs.

Weather, holiday and daily-update experiments are not automatically included:
additional factors must improve chronological validation. Churn is a weak
headline because customer coverage is incomplete. Inventory savings require
product demand, stock availability, shelf life and waste costs; this forecast
does not supply those. Garden's cost finding supports an invoice/allocation
audit, not a claim of recoverable cash or poor management.

**Next:** run `13_week_ahead_validation.ipynb` independently. Evidence is saved
under `artifacts/week_ahead_forecasting/` for inspection without the private CSV.